# Spectral probe (D244, D270)

How well can a linear probe read the bands of the input spectrum from the stored SSL representations,
compared with the FFT of the same windows?

* `results/<dataset>.csv`: test R² of a ridge probe per spectral target, for every run and the FFT
  reference (`probe_spectral.py`, D244, D246)
* `results/<dataset>_eta2.csv`: class contrast of each band, η² of its relative energy over the training
  windows (`band_eta2.py`, D270)

The spectra of the datasets with the bands overlaid: `band_spectra.ipynb`.
Run from the repo root folder (paths are relative to it). Clear the outputs before committing.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr

if os.path.basename(os.getcwd()) == "spectral_probe":  # opened from its own folder: paths are relative to the repo root
    os.chdir(os.path.join("..", ".."))
RESULTS = "studies/spectral_probe/results"
DATASETS = ["forda", "cwru", "hust", "paderborn", "xjtu_sy"]

probe = {d: pd.read_csv(f"{RESULTS}/{d}.csv") for d in DATASETS if os.path.exists(f"{RESULTS}/{d}.csv")}
eta2 = {d: pd.read_csv(f"{RESULTS}/{d}_eta2.csv") for d in DATASETS if os.path.exists(f"{RESULTS}/{d}_eta2.csv")}
print("probe results:", list(probe), "| eta²:", list(eta2))

## Mean R² over the bands

Mean test R² over the 16 relative band energies (log share of the total; the target every dataset has),
per run, probes trained on all training windows (D246). Paderborn: mean over its 10 combinations. `floor_mean` = predicting the
training mean (negative when the test windows' mean differs from the training mean, e.g. other bearings).

In [ ]:
def band_mean(d, target="relative"):
    """Mean R² over the bands of one target kind, per name, probe and label fraction (mean over channels and
    Paderborn's combinations)."""
    r = d[d.target == target]
    return r.groupby(["name", "probe", "label_fraction"]).r2.mean().unstack()

for name, d in probe.items():
    print(f"=== {name}")
    t = band_mean(d)
    display(t.xs("ridge", level="probe").round(3))
    print("floor_mean:", t.xs("floor_mean", level="probe").mean().round(2).to_dict())

## R² per band

Ridge R² for each relative band energy (all training windows, D246), against the band's centre frequency (fraction of the
sample rate). FFT reference: black, dashed. Random-init encoders (`_random`, `_random_bn`): grey.

In [ ]:
def style(name):
    if name == "fft_reference":
        return dict(color="black", ls="--", lw=2, zorder=3)
    if "_random" in name:
        return dict(color="grey", lw=0.8, alpha=0.6)
    return dict(lw=1.2)

for dname, d in probe.items():
    r = d[(d.probe == "ridge") & (d.target == "relative") & (d.label_fraction == 1.0)]
    channels = r.channel.unique()
    fig, axes = plt.subplots(1, len(channels), figsize=(7 * len(channels), 4.5), squeeze=False)
    for ax, ch in zip(axes[0], channels):
        c = r[r.channel == ch]
        for name, g in c.groupby("name"):
            g = g.groupby("band")[["f_lo", "f_hi", "r2"]].mean()  # Paderborn: mean over combinations
            ax.plot((g.f_lo + g.f_hi) / 2, g.r2, label=name, **style(name))
        ax.set(title=f"{dname}: {ch}", xlabel="band centre (cycles per sample; Nyquist = 0.5)", ylabel="test R²",
               ylim=(max(-1, ax.get_ylim()[0]), 1.02))
        ax.axhline(0, color="k", lw=0.5)
    axes[0][-1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## Class contrast of the bands (D270)

Does a representation encode the bands that carry class information better than the bands that don't?
A band's class contrast is η² of its relative energy (log share of the total) over the training windows:
between-class over total sum of squares, from the signal and the class labels only (`band_eta2.py`).
0 = same mean in every class, 1 = constant within each class. Paderborn: η² of each combination's own
training windows (mean and range over the 10 combinations in the figure).

In [ ]:
for dname, e in eta2.items():
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for ch, c in e.groupby("channel"):
        g = c.groupby("band")
        centre = (g.f_lo.mean() + g.f_hi.mean()) / 2
        ax.plot(centre, g.eta2.mean(), marker="o", label=ch)
        if c.combination.notna().any():  # Paderborn: range over the combinations
            ax.fill_between(centre, g.eta2.min(), g.eta2.max(), alpha=0.2)
    ax.set(title=f"{dname}: class contrast per band", xlabel="band centre (cycles per sample; Nyquist = 0.5)",
           ylabel="η² (training windows)", ylim=(0, 1))
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

Each run's ridge R² per relative band energy (all training windows, D246) against the band's η², and
Spearman's ρ between the two over all bands and channels of a run. ρ > 0: the bands that separate the
classes are read better from the representation than the others. Paderborn: ρ per combination, mean over
the 10 (points in the figure: mean over combinations). FFT reference: black; random-init encoders: grey.
The FFT reference has a ρ of its own (bands with high η² can also be easier to read from the plain
spectrum), so a run's ρ is read against the FFT reference's ρ, not against 0.

In [ ]:
def with_eta2(dname):
    """Ridge R² of the relative band energies, each row with its band's η² (joined on combination, channel, band)."""
    r = probe[dname]
    r = r[(r.probe == "ridge") & (r.target == "relative") & (r.label_fraction == 1.0)]
    e = eta2[dname][["combination", "channel", "band", "eta2"]]
    # no combination (all but Paderborn) -> -1, so the keys can be joined; bands as integers in both tables
    r, e = (t.assign(combination=t.combination.fillna(-1).astype(int), band=t.band.astype(int)) for t in (r, e))
    m = r.merge(e, on=["combination", "channel", "band"], how="left", validate="many_to_one")
    if m.eta2.isna().any():
        raise ValueError(f"{dname}: probe rows without η² (run band_eta2.py with the same config)")
    return m

def rho_per_name(m):
    """Spearman's ρ between band R² and band η² per run name, over bands and channels; mean over combinations."""
    rows = [{"name": n, "combination": k, "rho": spearmanr(g.r2, g.eta2).statistic}
            for (n, k), g in m.groupby(["name", "combination"])]
    return pd.DataFrame(rows).groupby("name").rho.mean()

contrast = {d: with_eta2(d) for d in eta2 if d in probe}
for d, m in contrast.items():
    display(rho_per_name(m).round(2).to_frame(f"{d}: Spearman ρ, band R² vs η²"))

for dname, m in contrast.items():
    pts = m.groupby(["name", "channel", "band"])[["eta2", "r2"]].mean().reset_index()  # Paderborn: mean over combinations
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, g in pts.groupby("name"):
        s = style(name)
        ax.scatter(g.eta2, g.r2, s=14, marker="x" if name == "fft_reference" else "o",
                   color=s.get("color"), alpha=s.get("alpha", 0.9), label=name)
    ax.set(title=f"{dname}: band R² against class contrast", xlabel="η² of the band (training windows)",
           ylabel="ridge test R²", xlim=(0, 1))
    ax.set_ylim(max(-1, ax.get_ylim()[0]), 1.02)
    ax.axhline(0, color="k", lw=0.5)
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()